# CreditWise AI — ML Feature Engineering

## Objective

Prepare the cleaned credit application data for machine learning.

Target:
`approved`

Main objectives:
- Remove target leakage
- Remove metadata
- Remove redundant features
- Create meaningful financial features
- Separate numerical and categorical variables
- Split train/test data
- Build a reusable preprocessing pipeline

In [29]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

# 2. Define target

In [31]:
TARGET = "approved"

y = df[TARGET].copy()

print("Target distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(
    y.value_counts(normalize=True)
     .mul(100)
     .round(2)
)

Target distribution:
approved
0    7238
1    4762
Name: count, dtype: int64

Target percentage:
approved
0    60.32
1    39.68
Name: proportion, dtype: float64


# Create initial feature matrix

In [32]:
X = df.drop(columns=[TARGET]).copy()

print("Initial X shape:", X.shape)

Initial X shape: (12000, 43)


# Remove leakage and metadata

In [33]:
exclude_cols = [
    "model_target_approval",
    "applicant_id",
    "synthetic_enrichment",
    "synthetic_version"
]

X = X.drop(
    columns=exclude_cols,
    errors="ignore"
)

print("After removing leakage and metadata:", X.shape)

After removing leakage and metadata: (12000, 39)


# Remove manually created bands

In [34]:
band_cols = [
    "income_band",
    "dti_band",
    "credit_history_band",
    "employment_band",
    "payment_risk_indicator"
]

X = X.drop(
    columns=band_cols,
    errors="ignore"
)

print("After removing band features:", X.shape)

After removing band features: (12000, 34)


# Convert application date

In [35]:
df["application_date"] = pd.to_datetime(
    df["application_date"],
    errors="coerce"
)

print(df["application_date"].dtype)

datetime64[ns]


# Create date features

In [36]:
X["application_year"] = (
    df["application_date"].dt.year
)

X["application_month"] = (
    df["application_date"].dt.month
)

# Create financial features

In [37]:
# DTI percentage
X["dti_percent"] = (
    X["debt_to_income_ratio"] * 100
)
# Credit utilization percentage
X["utilization_percent"] = (
    X["credit_utilization_ratio"] * 100
)
# Requested credit limit relative to annual income
X["requested_limit_to_income"] = (
    X["requested_credit_limit"] /
    X["annual_income"].replace(0, np.nan)
)
# Disposable income ratio
X["disposable_income_ratio"] = (
    X["disposable_income"] /
    X["monthly_income"].replace(0, np.nan)
)
# Credit exposure relative to annual income
X["exposure_to_income"] = (
    X["total_credit_exposure"] /
    X["annual_income"].replace(0, np.nan)
)

# Remove redundant columns

In [38]:
X = X.drop(
    columns=[
        "monthly_income",
        "application_date"
    ],
    errors="ignore"
)

print("Shape after feature engineering:", X.shape)

Shape after feature engineering: (12000, 39)


# Replace infinite values

In [39]:
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

# Check missing values

In [40]:
missing_features = (
    X.isna()
     .sum()
     .sort_values(ascending=False)
)

missing_features[missing_features > 0]

Series([], dtype: int64)

# Feature inventory

In [41]:
feature_inventory = pd.DataFrame({
    "feature": X.columns,
    "dtype": X.dtypes.astype(str).values,
    "missing": X.isna().sum().values,
    "unique_values": X.nunique().values
})

feature_inventory

,feature,dtype,missing,unique_values
0,gender,object,0,2
1,age,int64,0,50
2,num_children,int64,0,5
3,family_size,int64,0,6
4,family_status,object,0,5
5,education_type,object,0,5
6,housing_type,object,0,5
7,own_car,object,0,3
8,own_property,object,0,2
9,income_type,object,0,5


# Check for leakage one more time

In [42]:
leakage_check = [
    col for col in [
        "approved",
        "model_target_approval",
        "applicant_id",
        "synthetic_enrichment",
        "synthetic_version"
    ]
    if col in X.columns
]

print("Potential leakage/metadata columns still present:")
print(leakage_check)

Potential leakage/metadata columns still present:
[]


# Display final feature list

In [43]:
print("Final feature count:", X.shape[1])

for i, col in enumerate(X.columns, start=1):
    print(f"{i:02d}. {col}")

Final feature count: 39
01. gender
02. age
03. num_children
04. family_size
05. family_status
06. education_type
07. housing_type
08. own_car
09. own_property
10. income_type
11. occupation_type
12. annual_income
13. years_employed
14. credit_history_months
15. existing_credit_lines
16. debt_to_income_ratio
17. late_payments_24m
18. has_email
19. has_work_phone
20. product_type
21. requested_credit_limit
22. application_channel
23. application_purpose
24. monthly_debt_obligation
25. monthly_expenses
26. total_outstanding_debt
27. current_credit_limit
28. current_credit_balance
29. credit_score
30. disposable_income
31. credit_utilization_ratio
32. total_credit_exposure
33. application_year
34. application_month
35. dti_percent
36. utilization_percent
37. requested_limit_to_income
38. disposable_income_ratio
39. exposure_to_income


# Separate numerical and categorical features

In [44]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 26
Number of categorical features: 11

Numerical features:
['age', 'num_children', 'family_size', 'annual_income', 'years_employed', 'credit_history_months', 'existing_credit_lines', 'debt_to_income_ratio', 'late_payments_24m', 'has_email', 'has_work_phone', 'requested_credit_limit', 'monthly_debt_obligation', 'monthly_expenses', 'total_outstanding_debt', 'current_credit_limit', 'current_credit_balance', 'credit_score', 'disposable_income', 'credit_utilization_ratio', 'total_credit_exposure', 'dti_percent', 'utilization_percent', 'requested_limit_to_income', 'disposable_income_ratio', 'exposure_to_income']

Categorical features:
['gender', 'family_status', 'education_type', 'housing_type', 'own_car', 'own_property', 'income_type', 'occupation_type', 'product_type', 'application_channel', 'application_purpose']


# Train/test split


In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (9600, 39)
X_test : (2400, 39)
y_train: (9600,)
y_test : (2400,)


# Verify target distribution

In [46]:
print("Training target distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training target distribution:
approved
0    60.31
1    39.69
Name: proportion, dtype: float64

Testing target distribution:
approved
0    60.33
1    39.67
Name: proportion, dtype: float64


# Re-identify feature types

In [47]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features: 26
Categorical features: 11


# Numerical preprocessing

In [48]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing

In [49]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

# Combine pipelines

In [50]:
preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_features
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_features
    )
])

In [60]:
import joblib
import os

# Make sure output folder exists
os.makedirs("../data/outputs", exist_ok=True)

# Re-fit preprocessor using the current sklearn version
preprocessor.fit(X_train)

# Save it again
joblib.dump(
    preprocessor,
    "../data/outputs/preprocessor.pkl"
)

print("Preprocessor rebuilt and saved successfully.")
print("Scikit-learn version:", __import__("sklearn").__version__)
print("Processed feature count:", len(preprocessor.get_feature_names_out()))

Preprocessor rebuilt and saved successfully.
Scikit-learn version: 1.7.2
Processed feature count: 77


# Fit ONLY on training data

In [51]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

# Check processed data

In [52]:
print("Original training features:", X_train.shape[1])
print("Processed training features:", X_train_processed.shape[1])

print("Original testing features:", X_test.shape[1])
print("Processed testing features:", X_test_processed.shape[1])

Original training features: 39
Processed training features: 77
Original testing features: 39
Processed testing features: 77


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Get final feature names

In [53]:
feature_names = preprocessor.get_feature_names_out()

print("Processed feature count:", len(feature_names))

print("\nFirst 30 processed features:")
print(feature_names[:30])

Processed feature count: 77

First 30 processed features:
['num__age' 'num__num_children' 'num__family_size' 'num__annual_income'
 'num__years_employed' 'num__credit_history_months'
 'num__existing_credit_lines' 'num__debt_to_income_ratio'
 'num__late_payments_24m' 'num__has_email' 'num__has_work_phone'
 'num__requested_credit_limit' 'num__monthly_debt_obligation'
 'num__monthly_expenses' 'num__total_outstanding_debt'
 'num__current_credit_limit' 'num__current_credit_balance'
 'num__credit_score' 'num__disposable_income'
 'num__credit_utilization_ratio' 'num__total_credit_exposure'
 'num__dti_percent' 'num__utilization_percent'
 'num__requested_limit_to_income' 'num__disposable_income_ratio'
 'num__exposure_to_income' 'cat__gender_F' 'cat__gender_M'
 'cat__family_status_Civil marriage' 'cat__family_status_Married']


# Final Module 5 checkpoint

In [54]:
print("=" * 50)
print("CREDITWISE ML FEATURE ENGINEERING SUMMARY")
print("=" * 50)

print("Original dataset:", df.shape)
print("Final feature dataset:", X.shape)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

print("Processed training data:", X_train_processed.shape)
print("Processed testing data :", X_test_processed.shape)

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Final processed features:", len(feature_names))

CREDITWISE ML FEATURE ENGINEERING SUMMARY
Original dataset: (12000, 44)
Final feature dataset: (12000, 39)
Training data: (9600, 39)
Testing data : (2400, 39)
Processed training data: (9600, 77)
Processed testing data : (2400, 77)
Numerical features: 26
Categorical features: 11
Final processed features: 77


In [58]:
import joblib

joblib.dump(X_train_processed, "../data/outputs/X_train_processed.pkl")
joblib.dump(X_test_processed, "../data/outputs/X_test_processed.pkl")
joblib.dump(y_train, "../data/outputs/y_train.pkl")
joblib.dump(y_test, "../data/outputs/y_test.pkl")
joblib.dump(feature_names, "../data/outputs/feature_names.pkl")

print("Processed ML datasets saved successfully.")

Processed ML datasets saved successfully.


In [59]:
import joblib
import os

os.makedirs(
    "../data/outputs",
    exist_ok=True
)

joblib.dump(
    preprocessor,
    "../data/outputs/preprocessor.pkl"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.
